Przed oddaniem zadania upewnij się, że wszystko działa poprawnie.
**Uruchom ponownie kernel** (z paska menu: Kernel$\rightarrow$Restart) a następnie
**wykonaj wszystkie komórki** (z paska menu: Cell$\rightarrow$Run All).

Upewnij się, że wypełniłeś wszystkie pola `TU WPISZ KOD` lub `TU WPISZ ODPOWIEDŹ`, oraz
że podałeś swoje imię i nazwisko poniżej:

In [ ]:
NAME = ""

---

# 2. Model TNC

Przedstawimy teraz zasadę działania modelu TNC, zaimplementujemy jego  najistotniejsze elementy oraz dokonamy ewaluacji modelu.

## 2.1. Idea modelu TNC
Dla ciągu czasowego $X \in \mathbb{R}^{D \times T}$ możemy określić okno przesuwne $X_{[t - \frac{\delta}{2}, t + \frac{\delta}{2}]}$ o długości $\delta$ wycentrowane wokół chwili $t$, które zawiera pomiary wszystkich cech w przedziale czasu $[t - \frac{\delta}{2}, t + \frac{\delta}{2}]$. Dla ułatwienia zapisu okno to będzie oznaczane jako $W_t$.

Celem metody uczenia reprezentacji będzie uzyskanie wektorowej reprezentacji dowolnego okna $W_t$.

### Temporalne sąsiedztwo
Dla okna $W_t$ zdefiniujmy jego temporalne sąsiedztwo $N_t$ jako zbiór wszystkich okien wycentrowanych w $t^*$, gdzie wartość ta jest próbkowana z rozkładu normalnego $t^* \sim \mathcal{N}(t, \eta \cdot \delta)$. Parametr $\eta$ definiuje szerokość / zakres sąsiedztwa temporalnego.

Wybór parametru $\eta$ może być oparty o wiedzę ekspercką, jednak w publikacji autorzy zaproponowali zastosowanie testu statystycznego **Augmented Dickey-Fuller (ADF)**, który będzie sprawdzać stacjonarność rozkładu próbek w sąsiedztwie temporalnym. Algorytm wyboru parametru zakłada, że rozpoczynamy od niewielkiej wartości, po czym iteracyjnie zwiększamy parametr $\eta$, powodując poszerzenie się temporalnego sąsiedztwa. Zwiększanie wartości następuje do momentu aż test statystyczny nie będzie w stanie odrzucić hipotezy zerowej.


### "Uczenie kontrastowe"
Okna w sąsiedztwie temporalnym możemy uznać jako podobne do obecnie rozważanego okna ciągu czasowego. Pozostaje zatem pytanie jak uzyskać przykłady negatywne, tak abyśmy mogli zastosować koncepcję uczenia kontrastowego. 

Moglibyśmy założyć, że wszystkie próbki (okna) poza temporalnym sąsiedztwem są przykładami negatywnymi. Może się jednak tak zdarzyć, że nawet bardzo odległe okno, jest w istocie podobne do obecnego (ma taką samą dynamikę zmian / pochodzi z tego samego rozkładu). W takiej sytuacji uczenie modelu nie byłoby efektywne. Autorzy proponują użycie podejścia **Positive-Unlabeled (PU) learning**, w którym mamy próbki bazowe, próbki pozytywne i próbki nieoznaczone. W naszym przykładzie próbkami nieoznaczonymi będą próbki spoza sąsiedztwa temporalnego.

Do próbek nieoznaczonych są przypisywane wagi $w$ (tutaj: hiperparametr metody), a każda próbka nieoznaczona jest traktowana jako połączenie próbki pozytywnej z wagą $w$ oraz próbki negatywnej z wagą $1 - w$. Zobaczymy to dalej w funkcji kosztu.

### Koder i dyskryminator
Model TNC będziemy uczyć za pomocą funkcji kosztu, która pozwala odróżniać reprezentacje próbek z tego samego sąsiedztwa temporalnego od próbek spoza sąsiedztwa.

Pierwszym elementem modelu TNC jest **koder** $\text{Enc}(W_t)$, które przekształca okno $W_t \in \mathbb{R}^{D \times \delta}$ w wektor reprezentacji $Z_t \in \mathbb{R}^M$.

Drugim elementem jest **dyskryminator** $\mathcal{D}(Z_t, Z)$ pozwalający aproksymować prawdopodobieństwo tego, że $Z$ jest reprezentacją okna w sąsiedztwie $N_t$. Innymi słowy, dla dwóch wektorów reprezentacji zwraca prawdopodobieństwo tego, że te wektory (okna) należą do tego samego sąsiedztwa temporalnego.

Przeanalizujmy rysunek przedstawiający zasadę działania modelu TNC:

![](./assets/tnc.png)

### Funkcja kosztu
W najlepszym przypadku dyskryminator powinien zwracać wartości bliskie 1 dla reprezentacji z tego samego temporalnego sąsiedztwa oraz wartość 0 w przeciwnym przypadku. Funkcja kosztu jest zdefiniowana następująco:

$$\mathcal{L} = -\mathbb{E}_{W_t\sim X}\left[ \mathbb{E}_{W_l \sim N_t}[\log \mathcal{D}(Z_t,Z_l)] +\mathbb{E}_{W_k \sim \bar{N}_t}[(1-w_t)\times \log(1 - \mathcal{D}(Z_t, Z_k)) + w_t\times \log\mathcal{D}(Z_t, Z_k)] \right]$$

## 2.2. Implementacja kodera
Ze względu na sekwencyjny charakter próbek w oknie, jako model kodera użyjemy dwukierunkowej sieci rekurencyjnego typu GRU, która na wyjściu cechy przekształci dodatkową warstwą liniową. Sieć rekurencyjna dostarcza nam stany ukryte w formie wektorów dla każdego punktu wejściowego w sekwencji, więc w celu otrzymania jednego wektora podsumowującego całą sekwencję należy dokonać odpowiedniej agregacji. W niniejszym koderze rozwarzymy dwie metody agregacji sekwencji stanów ukrytych:
* wybór ostatniego stanu ukrytego
* agregacja stanów ukrytych z wykorzystaniem mechanizmu uwagi (ang. _attention_) 

## Zadanie 2.1 (1.5 pkt)
Zaimplementuj dwie klasy dla dwóch niżej zdefiniowanych stanów ukrytych:

1. `LastHiddenAggreagation` (0.5 pkt) - metoda `forward(...)` zwraca ostatni stan ukryty z sekwencji (pamiętaj, że jest to sieć dwukierunkowa, więc ostatnie stany ukryte dla każdego z kierunku będą na przeciwnych końcach hidden states)
2. `AttentionAggregation` (1 pkt):
      * aplikuje [*scaled dot product attention*](https://arxiv.org/abs/1706.03762) oraz oblicza średnią z otrzymanych wektorów
      * podpowiedź: możesz wykorzystać `torch.nn.MultiHeadAttention` z parametrem `num_heads=1`, ale **należy mieć zrozumienie i intuicje, jak działa mechanizm uwagi i jak może pomagać w agregacji**

In [ ]:
import torch
from torch import nn, Tensor
import torch.nn.functional as F

class LastHiddenAggregation(nn.Module):
    def forward(self, rnn_output: Tensor) -> Tensor:
        """Aggregates information from sequence of vectors by taking last state.
        rnn_output: Tensor with dimensions [sequence_length, batch_size, hidden_dim]
        returns: Tensor with dimensions [batch_size, hidden_dim]
        """
        # Sieć jest dwukierunkowa, więc hidden_dim = 2 * hidden.
        # Wyjście GRU to konkatenacja [kierunek_wprzód; kierunek_wstecz].
        # Ostatni stan kierunku "wprzód" jest na końcu sekwencji (indeks -1),
        # natomiast ostatni stan kierunku "wstecz" jest na jej początku (indeks 0).
        half = rnn_output.shape[-1] // 2
        forward_last = rnn_output[-1, :, :half]
        backward_last = rnn_output[0, :, half:]
        return torch.cat([forward_last, backward_last], dim=-1)

class AttentionAggregation(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        # Pojedyncza głowa odpowiada klasycznemu scaled-dot-product attention.
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=1, batch_first=False
        )

    def forward(self, rnn_output: Tensor) -> Tensor:
        """Aggregates information from sequence of vectors with scaled-dot-product attention.
        rnn_output: Tensor with dimensions [sequence_length, batch_size, hidden_dim]
        returns: Tensor with dimensions [batch_size, hidden_dim]
        """
        # Self-attention: każdy element sekwencji "patrzy" na wszystkie pozostałe
        # i jest przeważany wagami uwagi (softmax z przeskalowanych iloczynów skalarnych).
        attn_output, _ = self.attention(rnn_output, rnn_output, rnn_output)
        # Agregacja sekwencji przeważonych wektorów poprzez uśrednienie po osi czasu.
        return attn_output.mean(dim=0)

## Zadanie 2.2 (1.5 pkt)
Zaimplementuj klasę `GRUEncoder` zgodnie z poniższymi wymaganiami:
- w metodzie `__init__()`:
  * utwórz instancję sieci GRU o wymiarze wejściowym `in_dim`, wymiarze ukrytym `hidden_dim`, która posiada jedną warstwę oraz jest dwukierunkowa (ustaw dodatkowo parametr `batch_first=False`)
  * utwórz instancję warstwy liniowej o odpowiednim wymiarze wejściowym oraz wyjściu o wymiarowości `emb_dim`
- w metodzie `_get_initial_hidden()` utwórz wektor zer jako początkowy stan ukryty
- w metodzie `forward()` przetwórz cechy okień podanych na wejściu i wyznacz wektory reprezentacji (używając sieci GRU a następnie warstwy liniowej)

In [ ]:
from typing import Literal

class GRUEncoder(nn.Module):
    def __init__(
        self, 
        agg: Literal["last", "attn"], 
        in_dim: int, 
        hidden_dim: int,                  
        emb_dim: int,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.emb_dim = emb_dim

        if agg == "last":
            self.agg = LastHiddenAggregation()
        elif agg == "attn":
            self.agg = AttentionAggregation(2*hidden_dim)
        else:
            raise ValueError(f"Invalid aggregation name")

        # Jednowarstwowa, dwukierunkowa sieć GRU.
        self.gru = nn.GRU(
            input_size=in_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False,
        )
        # Sieć dwukierunkowa -> wymiar wyjściowy GRU to 2*hidden_dim.
        self.linear = nn.Linear(2 * hidden_dim, emb_dim)
        
    def _get_initial_hidden(self, batch_size):
        # Kształt: [num_layers * num_directions, batch_size, hidden_dim] = [2, B, H].
        device = next(self.parameters()).device
        return torch.zeros(2, batch_size, self.hidden_dim, device=device)

    def forward(self, x):
        x = x.permute(2, 0, 1)

        batch_size = x.shape[1]
        h_0 = self._get_initial_hidden(batch_size)

        # rnn_output: [sequence_length, batch_size, 2*hidden_dim]
        rnn_output, _ = self.gru(x, h_0)
        # Agregacja sekwencji stanów ukrytych do jednego wektora na okno.
        aggregated = self.agg(rnn_output)
        # Rzutowanie na docelowy wymiar reprezentacji.
        emb = self.linear(aggregated)

        return emb

## 2.3. Implementacja dyskryminatora
W przypadku dyskryminatora porównamy dwa proste modele, tak aby nie ryzykować przeuczeniem tego komponentu, tj. iloczyn skalarny i wielowarstwowy perceptron (MLP).

## Zadanie 2.3 (2 pkt)
Zaimplementuj 2 modele dyskryminatora zgodnie z następującymi wymaganiami:

`DotProductDiscriminator` (1 pkt):
- w metodzie `forward()` zwróc iloczyn skalarny odpowiadających sobie reprezentacji (uprzednio znormalizuj reprezentacje do jednostkowej długości)

`MLPDiscriminator` (1 pkt):
- w metodzie `__init__()` utwórz wielowarstwowy perceptron z następującymi warstwami (`d` to wymiar wejściowy):
  * warstwa liniowa o rozmiarach 2*d na 4*d
  * aktywacja ReLU
  * dropout z prawdp. równym 0.5
  * kolejna warstwa liniowa o rozmiarze 4*d na 1 
- w metodzie `forward()` dokonaj konkatenacji reprezentacji `z` oraz `z_tilde` a następnie przekaż je do perceptrona
  

In [ ]:
class DotProductDiscriminator(nn.Module):
    def forward(self, z: Tensor, z_tilde: Tensor) -> Tensor:
        # Normalizacja do jednostkowej długości -> iloczyn skalarny = podobieństwo kosinusowe.
        z = F.normalize(z, dim=-1)
        z_tilde = F.normalize(z_tilde, dim=-1)
        # Iloczyn skalarny odpowiadających sobie reprezentacji (logit, bez sigmoidy).
        return (z * z_tilde).sum(dim=-1)

class MLPDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()

        # Wejściem jest konkatenacja dwóch reprezentacji, stąd wymiar 2*d.
        self.mlp = nn.Sequential(
            nn.Linear(2 * input_dim, 4 * input_dim),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(4 * input_dim, 1),
        )

    def forward(self, z: Tensor, z_tilde: Tensor) -> Tensor:
        # Konkatenacja reprezentacji i przetworzenie przez perceptron (zwraca logit).
        p = self.mlp(torch.cat([z, z_tilde], dim=-1))
        return p.view((-1,))

## 2.4. Implementacja funkcji kosztu
W celu implementacji funkcji kosztu możemy wspomóc się gotową funkcją binarnej entropii krzyżowej, zauważając że problem uczenia dyskryminatora (a zatem całego modelu TNC) sprowadza się do problemu klasyfikacji binarnej. **Należy zaimplementować funkcję kosztu w pełnej zgodności ze wzorem podanym w publikacji (i wspomnianym w niniejszym zeszycie)**

## Zadanie 2.4 (3 pkt)
Zaimplementuj moduł implementujący funkcję kosztu modelu TNC stosując następujące wskazówki/instrukcje:
- `z_t` to reprezentacje obecnie rozważanych okien, `z_p` to reprezentacje pozytywne, natomiast `z_n` to reprezentacje nieoznaczone
- najpierw wyznacz predykcje dyskryminatora dla pary `(z_t, z_p)` oraz `(z_t, z_n)`
- przygotuj etykiety (zera i jedynki) dla par podobnych oraz różnych
- oblicz funkcję binarnej entropii krzyżowej dla wyników dyskryminatora, pamiętaj, że:
  * parę `(z_t, z_p)` chcemy traktować jako pozytywną (klasa 1)
  * parę `(z_t, z_n)` chcemy traktować jako pozytywną z wagą `w` oraz jako negatywną (klasa 0) z wagą `1 - w`
- zaimplementuj metodę `_compute_accuracy()`, która sprawdzi czy decyzje dyskryminatora są właściwe (pary pozytywne mają prawdp. `> 0.5`, a pary negatwne `< 0.5`; pamiętaj o zastosowaniu funkcji sigmoid!); wyniki uśrednij uzyskując miarę accuracy

In [ ]:
from typing import Tuple


class TNCLossFunction(nn.Module):

    def __init__(
        self, 
        discriminator: Literal["dot", "mlp"], 
        emb_dim: int, 
        w: float
    ):
        super().__init__()

        if discriminator == "dot":
            self.discriminator = DotProductDiscriminator()
        elif discriminator == "mlp":
            self.discriminator = MLPDiscriminator(input_dim=emb_dim)
        else:
            raise ValueError(f"Invalid discriminator: {discriminator}")
            
        self.bce = torch.nn.BCEWithLogitsLoss()
        self.w = w

    def forward(
        self,
        z_t: Tensor,
        z_p: Tensor,
        z_n: Tensor,
    ) -> Tuple[Tensor, float]:
        # Logity dyskryminatora dla par pozytywnych i nieoznaczonych.
        d_p = self.discriminator(z_t, z_p)
        d_n = self.discriminator(z_t, z_n)

        # Etykiety: pary pozytywne -> 1, pary negatywne -> 0.
        ones = torch.ones_like(d_p)
        zeros = torch.zeros_like(d_n)

        # Składnik dla par z sąsiedztwa: -log D(z_t, z_p)  -> BCE z etykietą 1.
        loss_p = self.bce(d_p, ones)
        # Składnik PU dla par nieoznaczonych:
        #   -(1 - w) * log(1 - D)  -> BCE z etykietą 0, waga (1 - w)
        #   -w * log D             -> BCE z etykietą 1, waga w
        loss_n = (1 - self.w) * self.bce(d_n, zeros) + self.w * self.bce(d_n, ones)

        loss = loss_p + loss_n

        accuracy = self._compute_accuracy(d_p=d_p, d_n=d_n)
        
        return loss, accuracy

    @staticmethod
    def _compute_accuracy(d_p: Tensor, d_n: Tensor) -> float:
        # Logity -> prawdopodobieństwa.
        prob_p = torch.sigmoid(d_p)
        prob_n = torch.sigmoid(d_n)

        # Poprawna decyzja: pary pozytywne > 0.5, pary negatywne < 0.5.
        correct = torch.cat([prob_p > 0.5, prob_n < 0.5]).float()
        acc = correct.mean().item()
        
        return acc

## 2.5. Uruchomienie modelu TNC
Poniżej znajduje się kod pozwalający wyuczyć model TNC (implementacja w bibliotece PyTorch-Lightning). Ustawimy domyślny zbiór hiperparametrów i będziemy uczyć model przez 50 epok. Następnie zwizualizujemy otrzymane wektory reprezentacji i zastosujemy je w zadaniu klasyfikacji.

Moduł danych:

In [ ]:
import os

import pytorch_lightning as pl
from torch.utils.data import DataLoader

from src.dataset import TNCDataset


class TrainDataModule(pl.LightningDataModule):

    def __init__(
        self,
        mc_sample_size: int,
        window_size: int,
        batch_size: int,
    ):
        super().__init__()

        self.mc_sample_size = mc_sample_size
        self.window_size = window_size
        self.batch_size = batch_size

        self.dataset = torch.load(f="./data/simulated.pt")

    def train_dataloader(self) -> DataLoader:
        return self._dataloader("x_train")

    def val_dataloader(self) -> DataLoader:
        return self._dataloader("x_val", shuffle=False)

    def _dataloader(self, split: str, shuffle: bool = True) -> DataLoader:
        data = TNCDataset(
            x=self.dataset[split],
            mc_sample_size=self.mc_sample_size,
            window_size=self.window_size,
        )
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=int(os.environ.get("NUM_WORKERS", 0)),
        )


Model TNC:

In [ ]:
import numpy as np
from typing import Any

class TNCModel(pl.LightningModule):

    def __init__(self, hparams: dict[str, Any]):
        super().__init__()

        self.save_hyperparameters(hparams)

        self.encoder = GRUEncoder(
            agg=hparams["agg"],
            in_dim=hparams["in_dim"],
            hidden_dim=hparams["hidden_dim"],
            emb_dim=hparams["emb_dim"],
        )
        self._loss_fn = TNCLossFunction(
            discriminator=hparams["discriminator"],
            emb_dim=self.encoder.emb_dim,
            w=self.hparams["w"],
        )

        self.training_step_outputs = []
        self.validation_step_outputs = []
        
    def forward(self, x):
        return self.encoder(x)

    def training_step(self, batch, batch_idx):
        loss, acc = self._common_step(batch)
        return {"loss": loss, "acc": acc}

    def validation_step(self, batch, batch_idx):
        loss, acc = self._common_step(batch)
        return {"loss": loss, "acc": acc}

    def on_train_batch_end(self, outputs, batch, batch_idx) -> None:
        self.training_step_outputs.append(outputs)

    def on_validation_batch_end(self, outputs, batch, batch_idx) -> None:
        self.validation_step_outputs.append(outputs)
         
    def on_validation_epoch_start(self):
        avg_loss, avg_accs = self._summarize_outputs(self.training_step_outputs)
        self.training_step_outputs = []
        
        self.log("step", self.trainer.current_epoch)
        self.log("train/loss", avg_loss, on_epoch=True, on_step=False)
        self.log("train/acc", avg_accs, on_epoch=True, on_step=False)

    def on_validation_epoch_end(self):
        avg_loss, avg_accs = self._summarize_outputs(self.validation_step_outputs)
        self.validation_step_outputs = []
        
        self.log("step", self.trainer.current_epoch)
        self.log("val/loss", avg_loss, on_epoch=True, on_step=False)
        self.log("val/acc", avg_accs, on_epoch=True, on_step=False)

    def _common_step(self, batch):
        x_t, x_p, x_n, _ = batch
        mc_sample = x_p.shape[1]
        batch_size, f_size, len_size = x_t.shape

        x_p = x_p.reshape((-1, f_size, len_size))
        x_n = x_n.reshape((-1, f_size, len_size))
        x_t = x_t.repeat(mc_sample, 1, 1)

        z_t = self.encoder(x_t)
        z_p = self.encoder(x_p)
        z_n = self.encoder(x_n)

        loss, acc = self._loss_fn(z_t=z_t, z_p=z_p, z_n=z_n)

        return loss, acc

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )
        
    @staticmethod
    def _summarize_outputs(outputs):
        losses = [out["loss"].item() for out in outputs]
        accs = [out["acc"] for out in outputs]

        avg_loss = np.mean(losses)
        avg_accs = np.mean(accs)

        return avg_loss, avg_accs

Ustawienie domyślnych hiperparametrów:

In [ ]:
default_hparams = {
    "agg": "last",
    "discriminator": "dot",
    "in_dim": 3,
    "hidden_dim": 100,
    "emb_dim": 10,
    "window_size": 50,
    "mc_sample_size": 40,
    "w": 0.05,
    "num_epochs": 50,
    "lr": 1e-3,
    "weight_decay": 1e-3,
    "batch_size": 10,
    "name": "default",
}

Uczenie modelu TNC:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./data/logs --port 6006

In [ ]:
datamodule = TrainDataModule(
        mc_sample_size=default_hparams["mc_sample_size"],
        window_size=default_hparams["window_size"],
        batch_size=default_hparams["batch_size"],
)
datamodule.setup("train")

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger


def train(hparams):
    pl.seed_everything(42)

    datamodule = TrainDataModule(
        mc_sample_size=hparams["mc_sample_size"],
        window_size=hparams["window_size"],
        batch_size=hparams["batch_size"],
    )
    tnc = TNCModel(hparams)

    model_chkpt = ModelCheckpoint(
        dirpath=f"./data/checkpoints/{hparams['name']}/",
        filename="model",
        monitor="val/acc",
        mode="max",
        verbose=False,
    )
    trainer = pl.Trainer(
        logger=TensorBoardLogger(
            save_dir="./data/logs",
            name=f"TNC_{hparams['name']}",
            default_hp_metric=False,
        ),
        callbacks=[model_chkpt],
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        max_epochs=hparams["num_epochs"],
        accelerator="cpu", # change to "cuda", if want to train on GPU
    )

    trainer.fit(model=tnc, datamodule=datamodule)
    
    
train(hparams=default_hparams)

Ewaluacja modelu:
- wizualizacja wektorów ukrytych i surowych danych
- zadanie klasyfikacji stanu procesu generującego ciąg czasowy
- zadanie klasteryzacji wektorów reprezentacji

In [ ]:
from src.utils import visualize_embeddings
from src.evaluations import evaluate_model


def visualize(name: str):
    best_tnc_model = TNCModel.load_from_checkpoint(
        checkpoint_path=f"./data/checkpoints/{name}/model.ckpt"
    )

    best_encoder = best_tnc_model.encoder

    dataset = torch.load("./data/simulated.pt")

    fig = visualize_embeddings(
        x_all=dataset["x_all"],
        y_all=dataset["y_all"],
        encoder=best_encoder,
        window_size=best_tnc_model.hparams["window_size"],
    )
    return fig


visualize(default_hparams["name"])

In [ ]:
def evaluate_classification_clustering(name: str):
    best_tnc_model = TNCModel.load_from_checkpoint(
        checkpoint_path=f"./data/checkpoints/{name}/model.ckpt"
    )

    best_encoder = best_tnc_model.encoder

    dataset = torch.load("./data/simulated.pt")
    
    metrics = evaluate_model(
        dataset=dataset,
        encoder=best_encoder,
        window_size=best_tnc_model.hparams["window_size"],
    )
    
    return metrics


evaluate_classification_clustering(default_hparams["name"])

## Zadanie 2.5. Badanie wpływu metody agregacji oraz dyskryminatora TNC (1.5 pkt)
Korzystając z domyślnych hiperparametrów sprawdź, który z zaimplementowanych modułów agregacji oraz dyskryminatora dostarcza najlepszych rezultatów. Sprawdź wszystkie 4 kombinacje $\{\mathrm{last}, \mathrm{attn}\} \times \{\mathrm{dot}, \mathrm{mlp}\}$

In [ ]:
import pandas as pd

# Budżet czasu: bazowy run 50 epok ~ 10 min => ~12 s/epoka.
# 4 modele x 15 epok x ~12 s ~ 12 min (z zapasem < 15 min).
EPOCHS_25 = 15

# Sprawdzamy wszystkie 4 kombinacje agregacji i dyskryminatora,
# pozostałe hiperparametry zostawiamy domyślne.
results_25 = []
for agg in ["last", "attn"]:
    for discriminator in ["dot", "mlp"]:
        name = f"25_agg-{agg}_disc-{discriminator}"
        hparams = {
            **default_hparams,
            "agg": agg,
            "discriminator": discriminator,
            "name": name,
            "num_epochs": EPOCHS_25,
        }

        train(hparams)
        metrics = evaluate_classification_clustering(name)

        results_25.append({
            "agg": agg,
            "discriminator": discriminator,
            "auc_test": metrics["auc"]["test"],
            "silhouette": metrics["cluster"]["silhouette"],
            "davies_bouldin": metrics["cluster"]["davies_bouldin"],
        })

df_25 = pd.DataFrame(results_25)
# Wyższe AUC i Silhouette = lepiej; niższe Davies-Bouldin = lepiej.
df_25

## Zadanie 2.6. Badanie hiperparametrów metody TNC (1.5 pkt)
Korzystając z podanych funkcji `train()` oraz `evaluate_classification_clustering()` zbadaj następujące hiperparametry:
- `window_size` (zbadaj co najmniej 3 wartości)
- `w` (zbadaj co najmniej 3 wartości)

Jeśli uczenie modelu będzie zbyt czasochłonne można zredukować liczbę epok. Każdy z parametrów można zbadać osobno - nie ma potrzeby przeglądania ich przekroju. Analizy powinny być prowadzone na podstawie metryk: AUC (zbiór testowy) oraz metryki klasteryzacji (Silhouette oraz Davies-Bouldin). Utwórz tabelki, które podsumują wyniki eksperymentów. Skomentuj wyniki

In [ ]:
# Budżet czasu: 6 modeli, a window_size zmienia czas/epokę.
# Sumarycznie ~6 "domyślnych epok" pracy na epokę => ~72 s/epoka.
# 12 epok x ~72 s ~ 14.5 min (mieści się w ~15 min).
EPOCHS_26 = 12


def run_experiment(overrides: dict, name: str) -> dict:
    """Uczy model z podanymi nadpisaniami hiperparametrów i zwraca metryki ewaluacji."""
    hparams = {**default_hparams, **overrides, "name": name, "num_epochs": EPOCHS_26}
    train(hparams)
    metrics = evaluate_classification_clustering(name)
    return {
        "auc_test": metrics["auc"]["test"],
        "silhouette": metrics["cluster"]["silhouette"],
        "davies_bouldin": metrics["cluster"]["davies_bouldin"],
    }


# --- Badanie window_size (pozostałe hiperparametry domyślne) ---
window_results = []
for window_size in [20, 50, 80]:
    res = run_experiment({"window_size": window_size}, name=f"26_ws-{window_size}")
    window_results.append({"window_size": window_size, **res})

df_window = pd.DataFrame(window_results)


# --- Badanie wagi w (pozostałe hiperparametry domyślne) ---
w_results = []
for w in [0.01, 0.05, 0.2]:
    res = run_experiment({"w": w}, name=f"26_w-{w}")
    w_results.append({"w": w, **res})

df_w = pd.DataFrame(w_results)

print("Wpływ window_size:")
display(df_window)
print("\nWpływ wagi w:")
display(df_w)

## Podsumowanie wizualne: jak TNC działa na tym szeregu czasowym
Trzy wykresy opowiadają historię w trzech krokach:
1. **Problem** — wieloszeregowy sygnał jest kawałkami stacjonarny: zmienia reżim (stan) w czasie.
2. **Reprezentacja** — wektory TNC grupują się według ukrytego stanu (mimo że model uczył się bez etykiet).
3. **Reprezentacja w czasie** — przesuwając okno wzdłuż jednego szeregu widać, że reprezentacja "przeskakuje" dokładnie na granicach reżimów — to esencja idei temporalnego sąsiedztwa.

In [ ]:
import numpy as np
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", context="notebook")

# --- Wczytanie danych i wyuczonego (domyślnego) enkodera ---
viz_dataset = torch.load("./data/simulated.pt")
viz_model = TNCModel.load_from_checkpoint(
    checkpoint_path=f"./data/checkpoints/{default_hparams['name']}/model.ckpt"
)
encoder = viz_model.encoder.eval()
window_size = viz_model.hparams["window_size"]

x_all = viz_dataset["x_all"].numpy()              # [N, n_features, T]
y_all = np.round(viz_dataset["y_all"].numpy())    # [N, T] -> stany całkowite
N, n_features, T = x_all.shape

states = np.unique(y_all).astype(int)
palette = dict(zip(states, sns.color_palette("Set2", len(states))))
rng = np.random.default_rng(42)


def encode(windows: np.ndarray) -> np.ndarray:
    """windows: [batch, n_features, window_size] -> [batch, emb_dim]."""
    with torch.no_grad():
        x = torch.tensor(windows, dtype=torch.float32)
        return encoder(x).cpu().numpy()


# =========================================================
# WYKRES 1: surowy sygnał wieloszeregowy z zaznaczonymi reżimami (stanami)
# =========================================================
series_idx = 0
x = x_all[series_idx]                       # [n_features, T]
y = y_all[series_idx].astype(int)           # [T]
time = np.arange(T)

df_long = (
    pd.DataFrame({f"cecha {f}": x[f] for f in range(n_features)})
    .assign(t=time)
    .melt(id_vars="t", var_name="cecha", value_name="wartość")
)

fig1, ax1 = plt.subplots(figsize=(14, 4))
sns.lineplot(data=df_long, x="t", y="wartość", hue="cecha", linewidth=1.2, ax=ax1)

# cieniowanie tła wg stanu (granice tam, gdzie zmienia się y)
boundaries = np.where(np.diff(y) != 0)[0] + 1
seg_starts = np.concatenate([[0], boundaries])
seg_ends = np.concatenate([boundaries, [T]])
for s, e in zip(seg_starts, seg_ends):
    ax1.axvspan(s, e, color=palette[int(y[s])], alpha=0.15)

ax1.set_title("1) Sygnał wieloszeregowy z zaznaczonymi reżimami (kolor tła = stan)",
              fontweight="bold")
ax1.set_xlabel("czas")
plt.tight_layout()
plt.show()


# =========================================================
# WYKRES 2: reprezentacje TNC (PCA 2D) pokolorowane wg ukrytego stanu
# =========================================================
n_samples = 1000
si = rng.integers(0, N, n_samples)
st = rng.integers(0, T - window_size, n_samples)
windows = np.stack([x_all[i, :, s:s + window_size] for i, s in zip(si, st)])
win_state = np.array([y_all[i, s:s + window_size].mean() for i, s in zip(si, st)])
win_state = np.round(win_state).astype(int)

z = encode(windows)
z_2d = PCA(n_components=2).fit_transform(z)

fig2, ax2 = plt.subplots(figsize=(7, 6))
sns.scatterplot(x=z_2d[:, 0], y=z_2d[:, 1], hue=win_state,
                palette=palette, s=25, edgecolor="none", ax=ax2)
ax2.set_title("2) Reprezentacje TNC (PCA) — separacja wg stanu (uczenie bez etykiet!)",
              fontweight="bold")
ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
ax2.legend(title="stan")
plt.tight_layout()
plt.show()


# =========================================================
# WYKRES 3: trajektoria reprezentacji w czasie (PC1) wzdłuż jednego szeregu
# =========================================================
stride = max(1, window_size // 5)
starts = np.arange(0, T - window_size + 1, stride)
traj_windows = np.stack([x[:, s:s + window_size] for s in starts])
z_traj = encode(traj_windows)
pc1 = PCA(n_components=1).fit_transform(z_traj)[:, 0]

centers = starts + window_size // 2
center_state = y[centers]

fig3, ax3 = plt.subplots(figsize=(14, 4))
sns.lineplot(x=centers, y=pc1, color="0.6", linewidth=1, ax=ax3)
sns.scatterplot(x=centers, y=pc1, hue=center_state, palette=palette, s=45,
                edgecolor="none", ax=ax3)
ax3.set_title("3) Trajektoria reprezentacji TNC w czasie (PC1) — skoki na granicach reżimów",
              fontweight="bold")
ax3.set_xlabel("czas (środek okna)"); ax3.set_ylabel("PC1 reprezentacji")
ax3.legend(title="stan")
plt.tight_layout()
plt.show()

In [ ]:
# --- Porównanie wyników Zadania 2.6 (window_size oraz w) ---
# Wyższe = lepiej dla AUC i Silhouette; niższe = lepiej dla Davies-Bouldin.
metrics = ["auc_test", "silhouette", "davies_bouldin"]
lower_is_better = {"davies_bouldin"}

experiments = [("window_size", df_window), ("w", df_w)]

fig, axes = plt.subplots(len(experiments), len(metrics), figsize=(15, 8))

for row, (param, df) in enumerate(experiments):
    x_labels = df[param].astype(str)
    for col, metric in enumerate(metrics):
        ax = axes[row, col]
        vals = df[metric].values
        best = vals.argmin() if metric in lower_is_better else vals.argmax()
        colors = ["#d62728" if i == best else "#7fa8d1" for i in range(len(vals))]

        ax.bar(x_labels, vals, color=colors)
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

        arrow = "↓" if metric in lower_is_better else "↑"
        ax.set_title(f"{metric} ({arrow} = lepiej)", fontweight="bold")
        ax.set_xlabel(param)
        ax.set_ylabel("")
        ax.margins(y=0.15)

fig.suptitle("Zadanie 2.6 — wpływ hiperparametrów (czerwony = najlepsza wartość)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Komentarz do wyników

**`window_size`.** Najlepszy wynik daje wartość **50** — to ona maksymalizuje zarówno AUC na zbiorze testowym (0.888), jak i Silhouette (0.211). Widać wyraźny kompromis: okno zbyt krótkie (20) nie zawiera wystarczającego kontekstu do rozpoznania reżimu, a okno zbyt długie (80) zaczyna "rozmywać" granice — często łączy w jednym oknie dwa różne stany, przez co jakość klasyfikacji wyraźnie spada (AUC 0.76). Niskie Davies-Bouldin dla 20 i 80 jest mylące: skupienia są ciasne, ale słabo odpowiadają rzeczywistym stanom (niższe AUC), więc nie należy traktować go w izolacji.

**`w`.** Wpływ jest niewielki — wszystkie trzy wartości dają AUC ≈ 0.88. Najlepsze AUC daje **0.01** (0.889), ale różnica względem 0.05 jest pomijalna. Co ciekawe, większe `w = 0.20` poprawia metryki klasteryzacji (Silhouette 0.230, najniższe Davies-Bouldin 1.453) kosztem minimalnego spadku AUC. Jest to spójne z rolą `w` w PU-learning: większa waga traktuje część próbek nieoznaczonych jako pozytywne, co "ściąga" reprezentacje podobnych okien bliżej siebie (lepsze skupienia), ale przy zbyt dużej wartości grozi wprowadzeniem fałszywych pozytywów.

**Wniosek.** Najlepsza ogólna konfiguracja to **`window_size = 50`** z **`w` w okolicy 0.01–0.05** (najlepsze AUC). Jeśli priorytetem jest jakość klasteryzacji, warto rozważyć `w = 0.20`.